In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf
from keras.applications.resnet50 import preprocess_input
import os
from pathlib import Path
from keras.preprocessing import image
from keras.applications.resnet50 import preprocess_input
import random
# Display
from IPython.display import Image, display
import matplotlib.cm as cm
from PIL import Image
from skimage import io
from warnings import filterwarnings
from tensorflow import io
from tensorflow import image
from PIL import Image
from imutils import paths
import math
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load Data

In [ ]:
train_dir = '/kaggle/input/100-bird-species/train'
test_dir = '/kaggle/input/100-bird-species/test'
valid_dir = '/kaggle/input/100-bird-species/valid'

In [8]:
def count_subdirectories(path: str) -> int:
    """
    Counts the number of subdirectories in the given directory path.
    """
    dir_path = Path(path)
    subdirectories = [f for f in dir_path.iterdir() if f.is_dir()]
    return len(subdirectories)


num_subdirectories = count_subdirectories(train_dir)
print(f"Number of subdirectories in {train_dir}: {num_subdirectories}")

Number of subdirectories in /kaggle/input/100-bird-species/train: 525


#  **Visualization**

In [17]:
train_image_path_list = list(sorted(paths.list_images(train_dir)))
train_image_path_sample = random.sample(population=train_image_path_list, k=20)

def examine_images(images: list, pdf_filename='examined_images.pdf'):
    num_images = len(images)
    num_rows = int(math.ceil(num_images / 5))
    num_cols = 5

    with PdfPages(pdf_filename) as pdf:
        fig, axs = plt.subplots(num_rows, num_cols, figsize=(30, 30), tight_layout=True)
        axs = axs.ravel()

        for i, image_path in enumerate(images[:num_images]):
            image = Image.open(image_path)
            label = PurePath(image_path).parent.name
            axs[i].imshow(image)
            axs[i].set_title(f"Bird: {label}", fontsize=25)
            axs[i].axis('off')

        pdf.savefig()
        plt.close()

examine_images(train_image_path_sample, pdf_filename='examined_images.pdf')



In [ ]:
class_names = sorted(os.listdir(train_dir))

# You might want to exclude any non-directory files or hidden files
class_names = [class_name for class_name in class_names if os.path.isdir(os.path.join(train_dir, class_name))]
from wordcloud import WordCloud
class_names_wordcloud = " ".join(class_names)
wcloud = WordCloud(width=800, height=400, background_color="white").generate(class_names_wordcloud)

# Display the word cloud image using matplotlib
plt.figure(figsize=(10, 5))
plt.imshow(wcloud, interpolation="bilinear")


# Save the word cloud image as a PDF (after displaying)
plt.savefig("WORDcloud.pdf", format="pdf")
plt.show()

# **Data Augmentation**

In [9]:
image_size = 224
batch_size = 32

# Data augmentation for training data
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.experimental.preprocessing.RandomFlip("horizontal"),
    tf.keras.layers.experimental.preprocessing.RandomRotation(0.2),
    tf.keras.layers.experimental.preprocessing.RandomZoom(0.2),
    tf.keras.layers.experimental.preprocessing.RandomContrast(0.2),
])

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=(image_size, image_size),
    batch_size=batch_size,
    shuffle=True,
    seed=42,
    validation_split=0.005,
    subset="training",
    label_mode="int",
)

train_data = train_data.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)


val_data = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir,
    image_size=(image_size, image_size),
    batch_size=batch_size,
    shuffle=False,
    validation_split=0.99,
    subset="validation",
    label_mode="int",
)

Found 84635 files belonging to 525 classes.
Using 84212 files for training.
Found 2625 files belonging to 525 classes.
Using 2598 files for validation.


# Model

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, GlobalAveragePooling2D, Dropout, Flatten

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

In [ ]:
# Create a Sequential model
model = keras.Sequential()

# Add ResNet50 base model to the Sequential model
resnet_base = ResNet50(include_top=False,
                       pooling='avg',
                       input_shape=(image_size, image_size, 3),
                       weights='imagenet',
                       classes = 525)

model.add(resnet_base)

# Access the last layer of the ResNet50 base model
last_conv_layer = resnet_base.get_layer(index=-2)

print("Last Convolutional Layer Name:", last_conv_layer.name if last_conv_layer else "Not Found")

In [ ]:
model.summary()

In [ ]:
model.layers[0].trainable = False
model.layers[0].layers[-10].trainable = True  # Adjust the number of layers you want to fine-tune


In [ ]:
model.add(Dense(1024, activation='relu'))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(525, activation='softmax'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss = 'sparse_categorical_crossentropy',
              metrics=['accuracy'])

# **Training**

In [ ]:
history = model.fit(train_data,
                    epochs=50,
                    validation_data=val_data)

In [ ]:
model.save("model.h5")

In [ ]:
# Save the trained model using pickle
with open('trained_model.pkl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
# Save the trained TensorFlow model
model.save('trained_model_tf')

# Model Evaluation

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3))

# Plot the training accuracy with a solid line
plt.plot(history.history['accuracy'], label='Train', linestyle='-')

# Plot the validation accuracy with a dotted line
plt.plot(history.history['val_accuracy'], label='Val', linestyle='--')

plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')  # Place the legend at the bottom right

# Save the figure as a PDF
# Save the figure as a PDF
filename = "model_accuracy.pdf"
file_path = '/content/drive/MyDrive/Birds/'  # Replace with your desired path

plt.tight_layout()
figure = plt.gcf()  # Get the current figure
figure.savefig(f"{filename}.pdf", format="pdf")
# Display the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3))

# Plot the training accuracy with a solid line
plt.plot(history.history['loss'], label='Train', linestyle='-')

# Plot the validation accuracy with a dotted line
plt.plot(history.history['val_loss'], label='Val', linestyle='--')

plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='lower right')  # Place the legend at the bottom right

# Save the figure as a PDF
# Save the figure as a PDF
filename = "model_loss.pdf"
file_path = '/content/drive/MyDrive/Birds/'  # Replace with your desired path

plt.tight_layout()
figure = plt.gcf()  # Get the current figure
figure.savefig(f"{filename}.pdf", format="pdf")
# Display the plot
plt.show()


In [ ]:
test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(image_size, image_size),
    shuffle=False
)

In [ ]:
result = model.evaluate(test_data)
print("Test loss, Test accuracy : ", result)

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# Function to make predictions
def predict_and_evaluate(model, test_data):
    predictions = []
    true_labels = []

    for images, labels in test_data:
        true_labels.extend(labels.numpy())
        preds = model.predict(images)
        predictions.extend(np.argmax(preds, axis=1))

    # Convert lists to numpy arrays for easier handling
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Calculate precision, recall, and F1-score
    precision = precision_score(true_labels, predictions, average='weighted')
    recall = recall_score(true_labels, predictions, average='weighted')
    f1 = f1_score(true_labels, predictions, average='weighted')

    print("Precision: {:.2f}".format(precision))
    print("Recall: {:.2f}".format(recall))
    print("F1-score: {:.2f}".format(f1))

    return predictions, true_labels

# Get predictions and true labels, and evaluate the model
predictions, true_labels = predict_and_evaluate(model, test_data)

# Gradient Class Activstion Maps

In [ ]:
# Converting Images to array
def get_img_array(img_path, size):
    # `img` is a PIL image of size 299x299
    img = keras.preprocessing.image.load_img(img_path, target_size=size)
    # `array` is a float32 Numpy array of shape (299, 299, 3)
    array = keras.preprocessing.image.img_to_array(img)
    # We add a dimension to transform our array into a "batch"
    # of size (1, 299, 299, 3)
    array = np.expand_dims(array, axis=0)
    return array

In [ ]:
# Function to make Grad-CAM heatmap
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


In [ ]:
superimposed_images = []
labels = []

In [ ]:
# Function to display Grad-CAM with label
def display_gradcam(img_array, heatmap, label, alpha=0.4):
    img_array = np.expand_dims(img_array, axis=0)
    img_array = np.squeeze(img_array, axis=0)  #
    heatmap = np.uint8(255 * heatmap)
    jet = cm.get_cmap("jet", 256)
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    jet_heatmap = image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img_array.shape[2], img_array.shape[1]))
    jet_heatmap = image.img_to_array(jet_heatmap)

    superimposed_img = jet_heatmap * alpha + img_array[0]
    superimposed_img = image.array_to_img(superimposed_img)

    superimposed_images.append(superimposed_img)
    labels.append(label)
    #plt.imshow(superimposed_img)
    #plt.title(f'Label: {label}', fontsize=10)
    #plt.axis('off')


In [ ]:
# Get a list of subdirectories (bird species folders) in the test dataset
bird_species_folders = [f.path for f in os.scandir(test_dir) if f.is_dir()]

# Iterate over bird species folders
for bird_species_folder in bird_species_folders:
    # Use the folder name as the label
    label = os.path.basename(bird_species_folder)

    # Get a list of image files in the current bird species folder
    image_files = [f.path for f in os.scandir(bird_species_folder) if f.is_file()]

    # Choose a random image from the current folder
    random_image_path = random.choice(image_files)

    # Get the array representation of the image
    img_array = get_img_array(random_image_path, size=(224, 224))

    # Make Grad-CAM
    heatmap = make_gradcam_heatmap(img_array, resnet_base, last_conv_layer.name)

    # Display Grad-CAM with label
    display_gradcam(img_array=img_array, heatmap=heatmap, label=label)
    #plt.show()

In [ ]:
# Function to display images with labels and save to a PDF file
def display_images_with_labels_and_save(images, labels, max_images_per_row=8, pdf_filename="output.pdf"):
    num_images = len(images)
    num_rows = int(np.ceil(num_images / max_images_per_row))

    pdf_pages = matplotlib.backends.backend_pdf.PdfPages(pdf_filename)
    
    fig, axes = plt.subplots(num_rows, max_images_per_row, figsize=(15, 2 * num_rows))
    axes = axes.flatten()

    for i in range(num_images):
        axes[i].imshow(images[i])
        axes[i].set_title(labels[i], fontsize=10)
        axes[i].axis('off')

    # Remove empty subplots in the last row
    for j in range(num_images, num_rows * max_images_per_row):
        fig.delaxes(axes[j])

    plt.tight_layout()
    pdf_pages.savefig(fig)
    pdf_pages.close()

# Display images with labels and save to a PDF file
display_images_with_labels_and_save(superimposed_images, labels, pdf_filename="grad_cams_output.pdf")

# Predictions

In [ ]:
# Function to make predictions
def predict_and_plot(model, test_data):
    predictions = []
    true_labels = []

    for images, labels in test_data:
        true_labels.extend(labels.numpy())
        preds = model.predict(images)
        predictions.extend(np.argmax(preds, axis=1))

    return predictions, true_labels

# Get predictions and true labels
predictions, true_labels = predict_and_plot(model, test_data)

In [ ]:
# Function to plot images and labels
def plot_images(images, labels, predictions, class_names):
    plt.figure(figsize=(12, 12))
    for i in range(len(images)):
        plt.subplot(4, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"True: {class_names[labels[i]]}\nPred: {class_names[predictions[i]]}")
        plt.axis("off")
    plt.show()

In [ ]:
plot_images(images[:10], true_labels[:10], predictions[:10], class_names[:10])

# Classification Report, Confusion Matrix

In [ ]:
# Function to plot classification report and confusion matrix
def plot_metrics(true_labels, predictions, class_names):
    # Classification Report
    print("Classification Report:")
    print(classification_report(true_labels, predictions, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(true_labels, predictions)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:
# Function to save figures in PDF format
def save_figures_as_pdf(figures, filenames):
    for figure, filename in zip(figures, filenames):
        figure.savefig(f"{filename}.pdf", format="pdf")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
images_to_save = next(iter(test_data))[0][:16]
true_labels_to_save = true_labels[:16]
predictions_to_save = predictions[:16]
class_names_to_save = class_names
plot_images(images_to_save, true_labels_to_save, predictions_to_save, class_names_to_save)

In [ ]:
import seaborn as sns
# Save the figures as PDF
figures = [plt.figure(num) for num in plt.get_fignums()]
figure_filenames = ["sample_images", "classification_report", "confusion_matrix"]
save_figures_as_pdf(figures, figure_filenames)

# Plot classification report and confusion matrix
df_report, cm = plot_metrics(true_labels, predictions, class_names)

# Save the classification report and confusion matrix as CSV files
df_report.to_csv("classification_report.csv", index=False)
np.savetxt("confusion_matrix.csv", cm, delimiter=",")